**IMPORTING LIBRARIES**

In [1]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

**LOADING AND INSPECTING DATASET**

In [2]:
def load_and_explore_dataset(file_path):
        print("-" * 60)
        print("LOAD AND EXPLORE DATE")
        print("-" * 60)

        df = pd.read_csv(file_path)
        print('shape of the dataset')
        print(df.shape)
        print('\ncheck missing')
        print(df.isnull().sum())
        print('\nFirst five row')
        print(df.head())
        print('\nDescriptive statistics')
        print(df.describe())
        print('\ndataset info')
        print(df.info())
        print('\nApproved Loan Distribution')
        print(df['loan_status'].value_counts())
        print('\nApproved Loan Percentage Distribution')
        print(df['loan_status'].value_counts(normalize=True))

        return df

**IDENTIFYING FEATURES**

In [3]:
#identifying features to use 
def identify_features(df):
    print("-" * 60)
    print("IDENTIFY FEATURES")
    print("-" * 60)

    #feature to drop
    feature_to_drop = ["loan_id"]

    #numerical columns
    num_features = [
        "no_of_dependents",
        "income_annum",
        "loan_amount",
        "loan_term",
        "credit_score",
        "residential_assets_value",
        "commercial_assets_value",
        "luxury_assets_value",
        "bank_asset_value"
    ]

    #category features
    cat_features = [
        "education",
        "self_employed"
    ]

    print("\nFeatures to drop:", feature_to_drop)
    print("\nNumerical features:", num_features)
    print("\nCategorical features:", cat_features)

    return feature_to_drop, num_features, cat_features

**CREATING DATA LABEL**

In [4]:
#preparing the data
def prepare_data(df, num_features, cat_features, feature_to_drop):
    print("-" * 60)
    print("PREPARING DATA")
    print("-" * 60)

    #drop unrelevant data
    df_model = df.drop(columns=feature_to_drop)

    #specify label
    #feature to use 
    X = df_model.drop("loan_status", axis=1)

    #target to use
    y = df_model["loan_status"]

    print("\nFeature Shape:", X.shape)
    print("\nTarget Shape:", y.shape)
    print("\nFeature columns list:", list(X.columns))

    return X, y, list(X.columns)

**PREPROCESSING PIPELINE**

In [5]:
#creating pipeline
def create_preprocessing_pipeline(num_features, cat_features):
    print("-" * 60)
    print("CREATE PREPROCESSING PIPELINE")
    print("-" * 60)

    #num columns
    numerical_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    #category features
    category_pipeline = Pipeline([
        ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])

    #column preprocessing
    preprocessing = ColumnTransformer([
        ("num", numerical_pipeline, num_features),
        ("cat", category_pipeline, cat_features)
    ])

    return preprocessing

**SPLITING DATA: TRAIN AND TEST**

In [6]:
def split_data(X, y, test_size=0.2, random_state=42):
    print("-" * 60)
    print("SPLITTING THE DATA")
    print("-" * 60)

    # x and y train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print("\nTraining set size:", X_train.shape[0])
    print("\nTest set size:", X_test.shape[0])

    print("\nTraining Approval distribution:", y_train.value_counts())
    print("\nTest Approval distribution:", y_test.value_counts())

    return X_train, X_test, y_train, y_test

**MODEL PIPELINE**

In [7]:
def create_pipeline(preprocessor):
    print("\n" + "-" * 60)
    print("CREATING PIPELINE")
    print("-" * 60)

    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    print("\nModel Pipeline created successfully")

    return model_pipeline

**RMC MODEL PIPELINE - RANDOM FOREST CLASSIFIER**

In [8]:
def create_rmc_pipeline(preprocessor):
    print("\n" + "-" * 60)
    print("CREATING PIPELINE")
    print("-" * 60)

    rmc_model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    print("\nModel Pipeline created successfully")

    return rmc_model_pipeline

**TRAINING MODEL**

In [9]:
def train(model_pipeline, X_train, y_train):
    print("\n" + "-" * 60)
    print("TRAINING MODEL - LOGISTIC REGRESSION")
    print("-" * 60)

    model_pipeline.fit(X_train, y_train)

    print("\nModel train successfully")

    #get name steps from preprocessing
    all_feature_names = model_pipeline.named_steps[
            "preprocessor"
    ].get_feature_names_out()

    #coefficient
    try:
        coefficient = model_pipeline.named_steps["classifier"].coef_[0]

        print("0 & 1 Clases:", model_pipeline.named_steps["classifier"].classes_)

        print("\nTop 10 most importance features (by coefficient magnitude)")
        
        feature_importance = pd.DataFrame({
            "feature": all_feature_names,
            "coefficient": coefficient
        }).sort_values("coefficient", key=abs, ascending=False)
        print(feature_importance.head(10).to_string(index=False))
    except Exception as e:
        print(f"Cannot extract feature name {e}")

    return model_pipeline 

**TRAIN MODEL - RANDOM FOREST CALSSIFIER**

In [10]:
def train_rmc(rmc_model_pipeline, X_train, y_train):
    print("\n" + "-" * 60)
    print("TRAINING MODEL - RANDOM FOREST CLASSIFIER")
    print("-" * 60)

    rmc_model_pipeline.fit(X_train, y_train)

    print("\nModel train successfully")

    #get name steps from preprocessing
    all_feature_names = rmc_model_pipeline.named_steps[
            "preprocessor"
    ].get_feature_names_out()

    #coefficient
    try:
        importance = rmc_model_pipeline.named_steps["classifier"].feature_importances_
    
        print("\nTop 10 Most Important Features")
        
        print("0 & 1 Clases:", rmc_model_pipeline.named_steps["classifier"].classes_)
        
        feature_importance = pd.DataFrame({
            "feature": all_feature_names,
            "importance": importance
        }).sort_values("importance", ascending=False)
    
        print(feature_importance.head(10).to_string(index=False))
    except Exception as e:
        print(f"Cannot extract feature importance: {e}")

    return rmc_model_pipeline 

**EVALUATING MODEL**

In [11]:
def evaluate_model(model_pipeline, X_train, X_test, y_train, y_test):
    print("\n" + "-" * 60)
    print("EVALUATE MODEL - LOGISITC REGRESSION")
    print("-" * 60)

    #predicted model
    y_train_pred = model_pipeline.predict(X_train)
    y_test_pred = model_pipeline.predict(X_test)

    #predicted model proba
    y_train_prob = model_pipeline.predict_proba(X_train)[:,1]
    y_test_prob = model_pipeline.predict_proba(X_test)[:,1]

    #metrics 
    y_train_acc_score = accuracy_score(y_train, y_train_pred)
    y_test_acc_score = accuracy_score(y_test, y_test_pred)

    y_train_roc_auc_score = roc_auc_score(y_train, y_train_prob)
    y_test_roc_auc_score = roc_auc_score(y_test, y_test_prob)

    print("\nMETRICS EVALUATION")
    print(f"\nTrain accuracy score: {float(y_train_acc_score):.4f}")
    print(f"\nTest accuracy score: {float(y_test_acc_score):.4f}")
    print(f"\nTrain ROC score: {float(y_train_roc_auc_score):.4f}")
    print(f"\nTest ROC score: {float(y_test_roc_auc_score):.4f}")

    print("\nCLASSIFICATION REPORT ON TRAIN SIZE")
    print("\nClassififcation report\n", classification_report(y_train, y_train_pred, target_names=["Approved", "Rejected"]))

    print("\nCLASSIFICATION REPORT ON TEST SIZE")
    print("\nClassififcation report\n", classification_report(y_test, y_test_pred, target_names=["Approved", "Rejected"]))

    #cross validation
    cv_score = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring="roc_auc")
    print("\nCross Validation(5-fold)")
    print(f" ROC AUC: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    metrics = {
        "y_train_acc_score": y_train_acc_score,
        "y_test_acc_score": y_test_acc_score,
        "y_train_roc_auc_score": y_train_roc_auc_score,
        "y_test_roc_auc_score": y_test_roc_auc_score,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "y_train_prob": y_train_prob,
        "y_test_prob": y_test_prob,
        "cv_score": cv_score
    }

    return metrics

**EVALUATE RMC MODEL**

In [12]:
def evaluate_rmc_model(rmc_model_pipeline, X_train, X_test, y_train, y_test):
    print("\n" + "-" * 60)
    print("EVALUATE MODEL - RANDOM FOREST CLASSIFIER")
    print("-" * 60)

    #predicted model
    y_train_pred = rmc_model_pipeline.predict(X_train)
    y_test_pred = rmc_model_pipeline.predict(X_test)

    #predicted model proba
    y_train_prob = rmc_model_pipeline.predict_proba(X_train)[:,1]
    y_test_prob = rmc_model_pipeline.predict_proba(X_test)[:,1]

    #metrics 
    y_train_acc_score = accuracy_score(y_train, y_train_pred)
    y_test_acc_score = accuracy_score(y_test, y_test_pred)

    y_train_roc_auc_score = roc_auc_score(y_train, y_train_prob)
    y_test_roc_auc_score = roc_auc_score(y_test, y_test_prob)

    print("\nMETRICS EVALUATION")
    print(f"\nTrain accuracy score: {float(y_train_acc_score):.4f}")
    print(f"\nTest accuracy score: {float(y_test_acc_score):.4f}")
    print(f"\nTrain ROC score: {float(y_train_roc_auc_score):.4f}")
    print(f"\nTest ROC score: {float(y_test_roc_auc_score):.4f}")

    print("\nCLASSIFICATION REPORT ON TRAIN SIZE")
    print("\nClassififcation report\n", classification_report(y_train, y_train_pred, target_names=["Approved", "Rejected"]))

    print("\nCLASSIFICATION REPORT ON TEST SIZE")
    print("\nClassififcation report\n", classification_report(y_test, y_test_pred, target_names=["Approved", "Rejected"]))

    #cross validation
    cv_score = cross_val_score(rmc_model_pipeline, X_train, y_train, cv=5, scoring="roc_auc")
    print("\nCross Validation(5-fold)")
    print(f" ROC AUC: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    rmc_metrics = {
        "y_train_acc_score": y_train_acc_score,
        "y_test_acc_score": y_test_acc_score,
        "y_train_roc_auc_score": y_train_roc_auc_score,
        "y_test_roc_auc_score": y_test_roc_auc_score,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "y_train_prob": y_train_prob,
        "y_test_prob": y_test_prob,
        "cv_score": cv_score
    }

    return rmc_metrics

**SAVING MODEL ARTIFACT**

In [13]:
def save_model_artifact(model_pipeline, rmc_model_pipeline, featured_columns):
    print("\n" + "-" * 60)
    print("SAVING MODEL ARTIFACT")
    print("-" * 60)

    joblib.dump(model_pipeline, "loan_approval_prediction.pkl")
    print("Loan Approval model saved successfuly")

    joblib.dump(rmc_model_pipeline, "rmc_loan_approval_prediction.pkl")
    print("RMC Loan Approval model saved successfuly")
    
    joblib.dump(featured_columns, "loan_approval_featured_columns.pkl")
    print("Feature columns saved successfuly")

    print("\n" + "-" * 60)
    print("ALL MODEL ARTIFACT SAVED")
    print("-" * 60)

**PREDICTION**

In [14]:
def predict_approval(candidate_data):
    model_pipeline = joblib.load("loan_approval_prediction.pkl")
    featured_columns = joblib.load("loan_approval_featured_columns.pkl")

    customer_df = pd.DataFrame([candidate_data])

    for feature in featured_columns:
        if feature not in customer_df.columns:
            raise ValueError(f"Misssing require {feature}")
    customer_df = customer_df[featured_columns]

    prediction = model_pipeline.predict(customer_df)[0]
    probability = model_pipeline.predict_proba(customer_df)[0]

    result = {
        "prediction": "Rejected" if prediction == 1 else "Approved",
        "rejected_probability": probability[1],
        "approved_probability": probability[0],
        "chance": "High" if probability[0] >= 0.8 else "Medium" if probability[0] >= 0.5 else "Low"
    }

    return result

**RMC PREDICTION**

In [15]:
def rmc_predict_approval(candidate_data):
    rmc_model_pipeline = joblib.load("rmc_loan_approval_prediction.pkl")
    featured_columns = joblib.load("loan_approval_featured_columns.pkl")

    customer_df = pd.DataFrame([candidate_data])

    for feature in featured_columns:
        if feature not in customer_df.columns:
            raise ValueError(f"Misssing require {feature}")
    customer_df = customer_df[featured_columns]

    prediction = rmc_model_pipeline.predict(customer_df)[0]
    probability = rmc_model_pipeline.predict_proba(customer_df)[0]

    rmc_result = {
        "prediction": "Rejected" if prediction == 1 else "Approved",
        "rejected_probability": probability[1],
        "approved_probability": probability[0],
        "chance": "High" if probability[0] >= 0.8 else "Medium" if probability[0] >= 0.5 else "Low"
    }

    return rmc_result

**TESTING PREDICTION**

In [20]:
def test_predict():
    print("\n" + "-" * 60)
    print("TESTING PREDICTION")
    print("-" * 60)

    candidate_data = {
        "no_of_dependents": 2,
        "income_annum": 1000000,
        "loan_amount": 20000000,
        "loan_term": 5,
        "credit_score": 450,
        "residential_assets_value": 15000000,
        "commercial_assets_value": 0,
        "luxury_assets_value": 0,
        "bank_asset_value": 5000000,
        "education": "Graduate",
        "self_employed": "No"
    }

    candidate_data_1 = {
        "no_of_dependents": 0,
        "income_annum": 15500000,
        "loan_amount": 20000000,
        "loan_term": 7,
        "credit_score": 550,
        "residential_assets_value": 15000000,
        "commercial_assets_value": 10000000,
        "luxury_assets_value": 0,
        "bank_asset_value": 5000000,
        "education": "Graduate",
        "self_employed": "Yes"
    }

    print("\nCandidate 1")
    result = predict_approval(candidate_data)
    print("Prediction -", result["prediction"])
    print("Rejected Probability -", result["rejected_probability"])
    print("Approved Probability -", result["approved_probability"])
    print("Chances of Aprroval -", result["chance"])

    print("\nCandidate 2")
    result_1 = predict_approval(candidate_data_1)
    print("Prediction -", result_1["prediction"])
    print("Rejected Probability -", result_1["rejected_probability"])
    print("Approved Probability -", result_1["approved_probability"])
    print("Chances of Aprroval -", result_1["chance"])

**TESTING RMC PREDICTION**

In [21]:
def test_rmc_predict():
    print("\n" + "-" * 60)
    print("TESTING PREDICTION")
    print("-" * 60)

    candidate_data = {
        "no_of_dependents": 2,
        "income_annum": 1000000,
        "loan_amount": 20000000,
        "loan_term": 5,
        "credit_score": 450,
        "residential_assets_value": 15000000,
        "commercial_assets_value": 0,
        "luxury_assets_value": 0,
        "bank_asset_value": 5000000,
        "education": "Graduate",
        "self_employed": "No"
    }

    candidate_data_1 = {
        "no_of_dependents": 0,
        "income_annum": 15500000,
        "loan_amount": 20000000,
        "loan_term": 7,
        "credit_score": 550,
        "residential_assets_value": 15000000,
        "commercial_assets_value": 10000000,
        "luxury_assets_value": 0,
        "bank_asset_value": 5000000,
        "education": "Graduate",
        "self_employed": "Yes"
    }

    print("\nCandidate 1")
    result = rmc_predict_approval(candidate_data)
    print("Prediction -", result["prediction"])
    print("Rejected Probability -", result["rejected_probability"])
    print("Approved Probability -", result["approved_probability"])
    print("Chances of Aprroval -", result["chance"])

    print("\nCandidate 2")
    result_1 = rmc_predict_approval(candidate_data_1)
    print("Prediction -", result_1["prediction"])
    print("Rejected Probability -", result_1["rejected_probability"])
    print("Approved Probability -", result_1["approved_probability"])
    print("Chances of Aprroval -", result_1["chance"])

**MAIN FUNCTION**

In [22]:
def main():
    file_path = "cleaned_loan_approval_dataset.csv"

    #load dataset
    df = load_and_explore_dataset(file_path)

    #categorize features(columns)
    feature_to_drop, num_features, cat_features = identify_features(df)

    #prepare data
    X, y, featured_columns = prepare_data(df, num_features, cat_features, feature_to_drop)

    #preprocessing pipeline
    preprocessing = create_preprocessing_pipeline(num_features, cat_features)

    #spliting data
    X_train, X_test, y_train, y_test = split_data(X, y)

    #model pipeline
    model_pipeline = create_pipeline(preprocessing)

    #rmc model pipeline
    rmc_model_pipeline = create_rmc_pipeline(preprocessing)

    #train model
    model_pipeline = train(model_pipeline, X_train, y_train)

    #train rmc model
    rmc_model_pipeline = train_rmc(rmc_model_pipeline, X_train, y_train)
    
    #eveluate model
    metrics = evaluate_model(model_pipeline, X_train, X_test, y_train, y_test)

    #evaluate rmc model 
    rmc_metrics = evaluate_rmc_model(rmc_model_pipeline, X_train, X_test, y_train, y_test)
    
    #save artifact
    save_model_artifact(model_pipeline, rmc_model_pipeline, featured_columns)

    #testing
    test_predict()

    #testing rmc
    test_rmc_predict()

In [23]:
if __name__ == "__main__":
    main()

------------------------------------------------------------
LOAD AND EXPLORE DATE
------------------------------------------------------------
shape of the dataset
(4196, 13)

check missing
loan_id                     0
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
credit_score                0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64

First five row
   loan_id  no_of_dependents      education self_employed  income_annum  \
0        1                 2       Graduate            No       9600000   
1        2                 0   Not Graduate           Yes       4100000   
2        3                 3       Graduate            No       9100000   
3        4                 3       Graduate            No       8200000   
4        5     

C:\Users\MARYAM\Desktop\Loan-Approval-Prediction-Model\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\MARYAM\Desktop\Loan-Approval-Prediction-Model\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\MARYAM\Desktop\Loan-Approval-Prediction-Model\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\MARYAM\Desktop\Loan-Approval-Prediction-Model\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in c

Prediction - Approved
Rejected Probability - 0.19
Approved Probability - 0.81
Chances of Aprroval - High


C:\Users\MARYAM\Desktop\Loan-Approval-Prediction-Model\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
